# Figure5_CPVAE_advantage_genes_and_GO

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'CPVAE_advantage5'
outdir.mkdir(parents=True, exist_ok=True)
# Copy/regenerate top5 raw fitness landscapes from existing long table
raw = pd.read_csv(model_comparison_dir / 'CPVAE_advantage_genes_observed_vs_reconstructed_raw_fitness_landscapes_long.csv')
adv = pd.read_csv(model_comparison_dir / 'CPVAE_gene_reconstruction_error_advantage_over_PCA_CP_VAE.csv')
top5 = adv.sort_values('CPVAE_advantage_vs_best_other', ascending=False).head(5)['Gene'].tolist()
sub=raw[raw['Gene'].isin(top5)].copy()
models=['Observed','PCA','CP','VAE','CPVAE']
fig, axes=plt.subplots(len(top5),len(models),figsize=(13.8,2.05*len(top5)),squeeze=False)
im=None
for r,gene in enumerate(top5):
    gene_label=sub[sub['Gene'].eq(gene)]['Gene_label'].iloc[0]
    vals=sub[sub['Gene'].eq(gene)]['Raw_fitness'].values; vmax=np.nanquantile(np.abs(vals),0.98) or 1
    for c,model in enumerate(models):
        ax=axes[r,c]
        mat=sub[(sub['Gene'].eq(gene)) & (sub['Model'].eq(model))].pivot(index='Time',columns='Space',values='Raw_fitness').reindex(index=Full_Timepoints,columns=Spacepoints).values
        im=ax.imshow(mat,aspect='auto',cmap='coolwarm',vmin=-vmax,vmax=vmax)
        if r==0: ax.set_title(model,fontsize=10,pad=7)
        if c==0:
            a=adv.set_index('Gene').loc[gene,'CPVAE_advantage_vs_best_other']; ax.set_ylabel(f'{gene_label}\n{gene}\nadv={a:.2f}',fontsize=7)
        ax.set_xticks(range(len(Spacepoints))); ax.set_xticklabels(Spacepoints,rotation=45,ha='right',fontsize=6)
        ax.set_yticks(range(len(Full_Timepoints))); ax.set_yticklabels(Full_Timepoints,fontsize=6)
fig.suptitle('Top5 genes where CPVAE has largest reconstruction-error advantage',fontsize=12,y=0.985)
fig.subplots_adjust(left=0.14,right=0.91,top=0.91,bottom=0.08,hspace=0.72,wspace=0.28)
fig.colorbar(im,cax=fig.add_axes([0.93,0.18,0.014,0.66]),label='Beta(log2FC)')
save_pdf(fig,outdir/'Figure5_top5_CPVAE_advantage_raw_fitness_landscapes.pdf')
sub.to_csv(outdir/'Figure5_top5_CPVAE_advantage_raw_fitness_landscapes_long.csv',index=False)
# GO for top100 advantageous genes: reuse function style from script4 by executing script4-like minimal code
ann=load_annotation(); locus_to_vc=ann.set_index('locus_ID')['KEGG_VC_number'].dropna().astype(str).to_dict(); background=set(ann['locus_ID'].astype(str))
go=pd.read_csv(input_dir/'uniprot_vch_go_all.tsv',sep='\t')
rows=[]
for _,row in go.iterrows():
    vcs=re.findall(r'VC_?A?\d+', str(row.get('Gene Names','')))
    for ont,col in [('BP','Gene Ontology (biological process)'),('CC','Gene Ontology (cellular component)'),('MF','Gene Ontology (molecular function)')]:
        text=row.get(col,'')
        if pd.isna(text): continue
        for name,goid in re.findall(r'([^;\[]+)\s*\[(GO:\d{7})\]',str(text)):
            for vc in vcs: rows.append({'VC':vc.replace('VCA','VC_A'),'GO_ID':goid,'GO_term':name.strip(),'Ontology':ont})
t2g=pd.DataFrame(rows).drop_duplicates(); vc_to_locus={v:k for k,v in locus_to_vc.items()}; t2g['Gene']=t2g['VC'].map(vc_to_locus); t2g=t2g.dropna(subset=['Gene'])
universe=set(t2g['Gene']).intersection(background); genes=set(adv.sort_values('CPVAE_advantage_vs_best_other',ascending=False).head(100)['Gene']).intersection(universe)
res=[]; M=len(universe); N=len(genes)
for (goid,term,ont),ss in t2g.groupby(['GO_ID','GO_term','Ontology']):
    tg=set(ss['Gene']).intersection(universe); K=len(tg); k=len(tg.intersection(genes))
    if k<2: continue
    p=hypergeom.sf(k-1,M,K,N); fold=(k/N)/(K/M) if N>0 and K>0 and M>0 else np.nan; res.append({'Set':'CPVAE_top100_advantage_genes','GO_ID':goid,'GO_term':term,'Ontology':ont,'Overlap':k,'Term_size':K,'Input_size':N,'Fold_enrichment':fold,'P_value':p,'Genes':'/'.join(sorted(tg.intersection(genes)))})
res=pd.DataFrame(res).sort_values('P_value') if res else pd.DataFrame()
if not res.empty:
    m=len(res); res['FDR_BH']=(res['P_value']*m/(np.arange(m)+1)).clip(upper=1); res['minus_log10_FDR']=-np.log10(res['FDR_BH'].replace(0,np.nextafter(0,1))); res['log10_Fold_enrichment']=np.log10(res['Fold_enrichment'].replace(0,np.nan))
res.to_csv(outdir/'Figure5_CPVAE_top100_advantage_genes_GO_enrichment.csv',index=False)
plot=res[(res['log10_Fold_enrichment'] > 0) & (res['P_value'] < 0.05)].sort_values('FDR_BH').head(20).iloc[::-1] if not res.empty else res
fig,ax=plt.subplots(figsize=(9.8,max(4,0.30*len(plot))))
if not plot.empty:
    plot = plot.copy()
    y=np.arange(len(plot))
    sizes = 35 + 24 * plot['Overlap'].astype(float)
    sc=ax.scatter(
        plot['log10_Fold_enrichment'], y,
        s=sizes,
        c=plot['minus_log10_FDR'],
        cmap='viridis', alpha=0.88,
        edgecolor='black', linewidth=0.25
    )
    ax.set_yticks(y); ax.set_yticklabels(plot['GO_term'],fontsize=7)
    ax.set_xlabel('log10(Fold enrichment)')
    ax.set_title('GO enrichment of CPVAE top100 advantage genes',fontsize=11,pad=9)
    cax = fig.add_axes([0.920, 0.43, 0.018, 0.34])
    cbar=fig.colorbar(sc, cax=cax)
    cbar.set_label('-log10(FDR)', fontsize=8)
    cbar.ax.tick_params(labelsize=7, length=2)
    count_values=sorted(plot['Overlap'].dropna().astype(int).unique().tolist())
    if len(count_values)>3:
        count_legend_values=[count_values[0], count_values[len(count_values)//2], count_values[-1]]
    else:
        count_legend_values=count_values
    handles=[plt.scatter([], [], s=35+24*v, color='#6b7280', alpha=0.75, edgecolor='black', linewidth=0.25) for v in count_legend_values]
    lax = fig.add_axes([0.892, 0.22, 0.075, 0.15])
    lax.axis('off')
    lax.legend(
        handles, [str(v) for v in count_legend_values], title='Count',
        frameon=False, fontsize=7, title_fontsize=7.5,
        loc='center', borderaxespad=0.0,
        handletextpad=0.4, labelspacing=0.25, handlelength=1.0
    )
else:
    ax.text(0.5,0.5,'No GO terms with overlap >= 2',ha='center',va='center'); ax.axis('off')
fig.subplots_adjust(left=0.42,right=0.86,top=0.90,bottom=0.12); save_pdf(fig,outdir/'Figure5_CPVAE_top100_advantage_genes_GO_enrichment.pdf')
print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/CPVAE_advantage5
